1. Install libraries

In [1]:
!pip install -q \
    transformers==4.41.2 \
    sentence-transformers==2.7.0 \
    accelerate==0.30.1 \
    peft==0.11.1 \
    sentencepiece \
    faiss-cpu

In [2]:
import re
import faiss
import numpy as np
import transformers

from google.colab import files
from sentence_transformers import SentenceTransformer
from transformers import pipeline

2. Upload Culpeper text

In [3]:
from google.colab import files

uploaded = files.upload()

filename = list(uploaded.keys())[0]

with open(filename, "r", encoding="utf-8") as f:
    text = f.read()

print(text[:1000])

Saving Culpeper_Gutenberg_X.txt to Culpeper_Gutenberg_X (1).txt
The Project Gutenberg eBook of The Complete Herbal
    
This eBook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this eBook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.

Title: The Complete Herbal

Author: Nicholas Culpeper


        
Release date: July 24, 2015 [eBook #49513]
                Most recently updated: October 24, 2024

Language: English

Other information and formats: www.gutenberg.org/ebooks/49513

Credits: Produced by Chris Curnow, Emmy and the Online Distributed
        Proofreading Team at http://www.pgdp.net (This file was
        produced from images generously made available by 

3. Clean the text

In [4]:
# Removing the text added by Gutenberg Project
start_marker = "THE COMPLETE HERBAL"
end_marker = "*** END OF THE PROJECT GUTENBERG EBOOK THE COMPLETE HERBAL ***"

start_index = text.find(start_marker)
end_index = text.find(end_marker)

if start_index == -1:
    raise ValueError("Start marker was not found.")

if end_index == -1:
    raise ValueError("End marker was not found.")

clean_text = text[start_index:end_index].strip()

In [5]:
# Checking the last step
print(clean_text[:500])
print(clean_text[-500:])

THE COMPLETE HERBAL ***
[Transcriber's Note: As with any medicinal work first published in the
1600s and rewritten countless times, it should go without saying to not
attempt these recipes. Just in case, the transcriber has now said it.
Also, many and varied were the printing and publishing anomalies, for a
more complete explanation, see the extensive notes collected at the end
of this text.]



[Illustration: NICHOLAS CULPEPER, M.D.

Author of the Family Herbal.]

[Illustration: RED LION HOUSE,
and sweating the last)

Page 396, “int*” changed to “into” (of sweet Almonds into)

Page 399, “fourth” repeated under “Hot in the first degree” under
“second”. The first “fourth” was changed to “third” (—— —— third
degree, ib.)

Page 401, page number added to entry for “Tamarisk Tree.”

Page 402, “Yellow-water Flag” changed to “Yellow Water-flag” (Yellow
Water-flag)

Page 402, “Termintil” changed to “Tormentil” (Measles. Tormentil, 184)

Page 402, page number added to entry for “Ladies’ Mantle.

4. Split text into chunks

In [6]:
# Choose the marker where the second part begins
split_marker = "DIRECTIONS FOR MAKING SYRUPS, CONSERVES,"
split_index = clean_text.find(split_marker)

if split_index == -1:
    raise ValueError("The beginning of the pharmaceutical section was not found.")

part1 = clean_text[:split_index]
part2 = clean_text[split_index:]

In [8]:
print(split_index)
print(clean_text[split_index:split_index+500])

733784
DIRECTIONS FOR MAKING SYRUPS, CONSERVES,

_&c._ _&c._


HAVING in divers places of this Treatise promised you the way of making
Syrups, Conserves, Oils, Ointments, &c., of herbs, roots, flowers, &c.
whereby you may have them ready for your use at such times when they
cannot be had otherwise; I come now to perform what I promised, and you
shall find me rather better than worse than my word.

That this may be done methodically, I shall divide my directions into
two grand sections, and each section


In [9]:
print("Part 1 length:", len(part1))
print("Part 2 length:", len(part2))

Part 1 length: 733784
Part 2 length: 740225


In [10]:
herb_pattern = r'\n\s*\n(?= {4}[A-Z][A-Z\s,;\-’\.]+\.?\n\n)'

herb_chunks = re.split(herb_pattern, part1)
herb_chunks = [chunk.strip() for chunk in herb_chunks if len(chunk.strip()) > 100]

print("Herb chunks:", len(herb_chunks))

Herb chunks: 329


In [11]:
chunk_size = 1000
overlap = 200

pharma_chunks = []

for i in range(0, len(part2), chunk_size - overlap):
    chunk = part2[i:i + chunk_size]
    if len(chunk.strip()) > 100:
        pharma_chunks.append(chunk.strip())

print("Pharmaceutical chunks:", len(pharma_chunks))

Pharmaceutical chunks: 926


In [12]:
chunk_data = []

for i, chunk in enumerate(herb_chunks):
    chunk_data.append({
        "chunk_id": i,
        "section": "herbs",
        "text": chunk
    })

for i, chunk in enumerate(pharma_chunks):
    chunk_data.append({
        "chunk_id": len(chunk_data),
        "section": "pharmaceutical_products",
        "text": chunk
    })

print("Total chunks:", len(chunk_data))
print(chunk_data[0])
print(chunk_data[250])
print(chunk_data[400])

Total chunks: 1255
{'chunk_id': 0, 'section': 'herbs', 'text': "THE COMPLETE HERBAL ***\n[Transcriber's Note: As with any medicinal work first published in the\n1600s and rewritten countless times, it should go without saying to not\nattempt these recipes. Just in case, the transcriber has now said it.\nAlso, many and varied were the printing and publishing anomalies, for a\nmore complete explanation, see the extensive notes collected at the end\nof this text.]\n\n\n\n[Illustration: NICHOLAS CULPEPER, M.D.\n\nAuthor of the Family Herbal.]\n\n[Illustration: RED LION HOUSE, SPITALFIELDS\n\nIN WHICH CULPEPER LIVED, STUDIED AND DIED]\n\n\n\n  THE\n\n  COMPLETE HERBAL;\n\n  TO WHICH IS NOW ADDED, UPWARDS OF\n\n  ONE HUNDRED ADDITIONAL HERBS,\n\n  WITH A DISPLAY OF THEIR\n\n  Medicinal and Occult Qualities\n\n  PHYSICALLY APPLIED TO\n\n  THE CURE OF ALL DISORDERS INCIDENT TO MANKIND:\n\n  TO WHICH ARE NOW FIRST ANNEXED, THE\n\n  ENGLISH PHYSICIAN ENLARGED,\n\n  AND\n\n  KEY TO PHYSIC.\n\n  W

5. Creating embeddings

In [13]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
texts = [item["text"] for item in chunk_data]

embeddings = model.encode(
    texts,
    show_progress_bar=True
)

Batches:   0%|          | 0/40 [00:00<?, ?it/s]

In [15]:
print(type(embeddings))
print(embeddings.shape)

<class 'numpy.ndarray'>
(1255, 384)


In [16]:
print(embeddings[250][:20])

[-0.00014991 -0.01465367  0.01729498 -0.0283989   0.00806678  0.02899372
  0.05522426 -0.05600992  0.04056588  0.00597305 -0.01691249 -0.04596281
 -0.02345731 -0.00704733 -0.06130128  0.10619138  0.01166929 -0.03543421
 -0.02396827 -0.00780899]


In [17]:
# Checking where the chunks are stored
print(type(texts))
print(type(texts[0]))
print(texts[0])

<class 'list'>
<class 'str'>
THE COMPLETE HERBAL ***
[Transcriber's Note: As with any medicinal work first published in the
1600s and rewritten countless times, it should go without saying to not
attempt these recipes. Just in case, the transcriber has now said it.
Also, many and varied were the printing and publishing anomalies, for a
more complete explanation, see the extensive notes collected at the end
of this text.]



[Illustration: NICHOLAS CULPEPER, M.D.

Author of the Family Herbal.]

[Illustration: RED LION HOUSE, SPITALFIELDS

IN WHICH CULPEPER LIVED, STUDIED AND DIED]



  THE

  COMPLETE HERBAL;

  TO WHICH IS NOW ADDED, UPWARDS OF

  ONE HUNDRED ADDITIONAL HERBS,

  WITH A DISPLAY OF THEIR

  Medicinal and Occult Qualities

  PHYSICALLY APPLIED TO

  THE CURE OF ALL DISORDERS INCIDENT TO MANKIND:

  TO WHICH ARE NOW FIRST ANNEXED, THE

  ENGLISH PHYSICIAN ENLARGED,

  AND

  KEY TO PHYSIC.

  WITH

  RULES FOR COMPOUNDING MEDICINE ACCORDING TO THE TRUE SYSTEM OF NATURE.



In [18]:
print(type(chunk_data))
print(type(chunk_data[0]))
print(chunk_data[0])

<class 'list'>
<class 'dict'>
{'chunk_id': 0, 'section': 'herbs', 'text': "THE COMPLETE HERBAL ***\n[Transcriber's Note: As with any medicinal work first published in the\n1600s and rewritten countless times, it should go without saying to not\nattempt these recipes. Just in case, the transcriber has now said it.\nAlso, many and varied were the printing and publishing anomalies, for a\nmore complete explanation, see the extensive notes collected at the end\nof this text.]\n\n\n\n[Illustration: NICHOLAS CULPEPER, M.D.\n\nAuthor of the Family Herbal.]\n\n[Illustration: RED LION HOUSE, SPITALFIELDS\n\nIN WHICH CULPEPER LIVED, STUDIED AND DIED]\n\n\n\n  THE\n\n  COMPLETE HERBAL;\n\n  TO WHICH IS NOW ADDED, UPWARDS OF\n\n  ONE HUNDRED ADDITIONAL HERBS,\n\n  WITH A DISPLAY OF THEIR\n\n  Medicinal and Occult Qualities\n\n  PHYSICALLY APPLIED TO\n\n  THE CURE OF ALL DISORDERS INCIDENT TO MANKIND:\n\n  TO WHICH ARE NOW FIRST ANNEXED, THE\n\n  ENGLISH PHYSICIAN ENLARGED,\n\n  AND\n\n  KEY TO PHY

In [19]:
print(len(texts))
print(len(embeddings))

1255
1255


6. Building FAISS index

In [20]:
embeddings = np.array(embeddings).astype("float32")

In [21]:
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

In [22]:
index.add(embeddings)

In [23]:
print(index.ntotal)

1255


7. Search for a plant

In [24]:
query = "violet"

In [25]:
query_embedding = model.encode([query])

In [26]:
query_embedding = np.array(query_embedding).astype("float32")

In [27]:
k = 3

distances, indices = index.search(query_embedding, k)

In [28]:
print(indices)

[[318 788 787]]


In [29]:
for idx in indices[0]:
    print("=" * 60)
    print(chunk_data[idx]["text"][:1000])

VIOLETS.

BOTH the tame and the wild are so well known, that they need no
description.

_Time._] They flower until the end of July, but are best in March, and
the beginning of April.

_Government and virtues._] They are a fine pleasing plant of Venus,
of a mild nature, no way harmful. All the Violets are cold and moist
while they are fresh and green, and are used to cool any heat,
or distemperature of the body, either inwardly or outwardly, as
inflammations in the eyes, in the matrix or fundament, in imposthumes
also, and hot swellings, to drink the decoction of the leaves and
flowers made with water in wine, or to apply them poultice-wise to the
grieved places: it likewise eases pains in the head, caused through
want of sleep; or any other pains arising of heat, being applied in
the same manner, or with oil of roses. A dram weight of the dried
leaves or flower of Violets, but the leaves more strongly, doth purge
the body of choleric humours, and assuages the heat, being taken in a
dra

In [30]:
print(len(chunk_data))

1255


In [31]:
print(len(embeddings))

1255


8. Summarize retrieved passages

In [32]:
summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn"
)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [43]:
retrieved_text = ""

for idx in indices[0]:
    retrieved_text += chunk_data[idx]["text"] + "\n\n"

In [44]:
print(retrieved_text[:5000])

VIOLETS.

BOTH the tame and the wild are so well known, that they need no
description.

_Time._] They flower until the end of July, but are best in March, and
the beginning of April.

_Government and virtues._] They are a fine pleasing plant of Venus,
of a mild nature, no way harmful. All the Violets are cold and moist
while they are fresh and green, and are used to cool any heat,
or distemperature of the body, either inwardly or outwardly, as
inflammations in the eyes, in the matrix or fundament, in imposthumes
also, and hot swellings, to drink the decoction of the leaves and
flowers made with water in wine, or to apply them poultice-wise to the
grieved places: it likewise eases pains in the head, caused through
want of sleep; or any other pains arising of heat, being applied in
the same manner, or with oil of roses. A dram weight of the dried
leaves or flower of Violets, but the leaves more strongly, doth purge
the body of choleric humours, and assuages the heat, being taken in a
dra

In [45]:
def summarize_long_text(retrieved_text, chunk_size=3000):
    # Split text by rough character count (approx 600-700 words)
    text_chunks = [
        retrieved_text[i:i+chunk_size]
        for i in range(0, len(retrieved_text), chunk_size)]

    # Summarize each chunk
    intermediate_summaries = []

    for text_chunk in text_chunks:
        result = summarizer(
            text_chunk,
            max_length=130,
            min_length=30,
            do_sample=False,
            truncation=True
        )

        intermediate_summaries.append(result[0]['summary_text'])

    # Combine individual summaries into one text
    combined_text = " ".join(intermediate_summaries)

    # Summarize the summaries once more
    final_result = summarizer(
        combined_text,
        max_length=120,
        min_length=40,
        do_sample=False,
        truncation=True
    )

    return final_result[0]["summary_text"]

In [46]:
final_summary = summarize_long_text(retrieved_text)

print("=" * 60)
print("QUESTION")
print(question)

print("\nSUMMARY")
print(final_summary)

Your max_length is set to 120, but your input_length is only 114. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=57)


QUESTION
What does Culpeper say about violet?

SUMMARY
Violets are cold and moist while they are fresh and green. The leaves are used with other herbs to make plaistersand poultices to inflammations and swellings. Julep of Violets is made of the water of Violet flowers and sugar.


9. Create an AI query

In [47]:
def build_qa_context(indices, chunk_data, qa_model, max_context_tokens=380):
    selected_text = "\n\n".join(
        chunk_data[idx]["text"]
        for idx in indices[0][:3]
    )

    token_ids = qa_model.tokenizer.encode(
        selected_text,
        add_special_tokens=False,
        truncation=True,
        max_length=max_context_tokens
    )

    return qa_model.tokenizer.decode(
        token_ids,
        skip_special_tokens=True
    )

In [48]:
qa_model = pipeline(
    "text2text-generation",
    model="google/flan-t5-base"
)

In [49]:
def answer_question_with_context(question, context):
    prompt = f"""
Answer the question using only the passages below.
Do not add outside information.
If the passages do not provide an answer, say that the information was not found.

Question:
{question}

Passages:
{context}

Answer:
"""

    result = qa_model(
        prompt,
        max_new_tokens=160,
        do_sample=False,
        truncation=True
    )

    return result[0]["generated_text"]

In [50]:
question = "What does Culpeper say about violet?"

qa_context = build_qa_context(
    indices,
    chunk_data,
    qa_model,
    max_context_tokens=380
)

answer = answer_question_with_context(question, qa_context)

print("\nANSWER")
print(answer)


ANSWER
They are a fine pleasing plant of Venus, of a mild nature, no way harmful.


In [41]:
print("\nSOURCES")

for idx in indices[0]:
    title = chunk_data[idx]["text"].split("\n")[0]
    print("-", title)


SOURCES
1. VIOLETS. (herbs)
2. up close together into (pharmaceutical_products)
3. yrupus de Symphyto._ (pharmaceutical_products)


In [51]:
print("SUMMARY")
print(final_summary)

print("\nQUESTION-FOCUSED ANSWER")
print(answer)

SUMMARY
Violets are cold and moist while they are fresh and green. The leaves are used with other herbs to make plaistersand poultices to inflammations and swellings. Julep of Violets is made of the water of Violet flowers and sugar.

QUESTION-FOCUSED ANSWER
They are a fine pleasing plant of Venus, of a mild nature, no way harmful.
